In [1]:
import random
import re
from collections import defaultdict, Counter

# Texto de ejemplo
corpus = "el gato duerme en el sofá. el perro duerme en la alfombra. el gato juega con el perro."

# Preprocesar texto (opcionalmente puedes hacer más limpieza)
tokens = re.findall(r'\b\w+\b', corpus.lower())  # tokeniza y pasa a minúsculas

In [2]:
trigrams = [ (tokens[i], tokens[i+1], tokens[i+2]) for i in range(len(tokens) - 2) ]

In [3]:
trigrams

[('el', 'gato', 'duerme'),
 ('gato', 'duerme', 'en'),
 ('duerme', 'en', 'el'),
 ('en', 'el', 'sofá'),
 ('el', 'sofá', 'el'),
 ('sofá', 'el', 'perro'),
 ('el', 'perro', 'duerme'),
 ('perro', 'duerme', 'en'),
 ('duerme', 'en', 'la'),
 ('en', 'la', 'alfombra'),
 ('la', 'alfombra', 'el'),
 ('alfombra', 'el', 'gato'),
 ('el', 'gato', 'juega'),
 ('gato', 'juega', 'con'),
 ('juega', 'con', 'el'),
 ('con', 'el', 'perro')]

In [6]:
model = defaultdict(list)



In [7]:
model

defaultdict(list, {})

In [8]:
for w1, w2, w3 in trigrams:
    model[(w1, w2)].append(w3)

In [9]:
model

defaultdict(list,
            {('el', 'gato'): ['duerme', 'juega'],
             ('gato', 'duerme'): ['en'],
             ('duerme', 'en'): ['el', 'la'],
             ('en', 'el'): ['sofá'],
             ('el', 'sofá'): ['el'],
             ('sofá', 'el'): ['perro'],
             ('el', 'perro'): ['duerme'],
             ('perro', 'duerme'): ['en'],
             ('en', 'la'): ['alfombra'],
             ('la', 'alfombra'): ['el'],
             ('alfombra', 'el'): ['gato'],
             ('gato', 'juega'): ['con'],
             ('juega', 'con'): ['el'],
             ('con', 'el'): ['perro']})

In [11]:
def generar_texto(modelo, inicio=('el', 'gato'), longitud=10):
    resultado = [inicio[0], inicio[1]]
    for _ in range(longitud):
        estado_actual = (resultado[-2], resultado[-1])
        siguientes = modelo.get(estado_actual)
        if not siguientes:
            break
        siguiente_palabra = random.choice(siguientes)
        resultado.append(siguiente_palabra)
    return ' '.join(resultado)

In [12]:
print(generar_texto(model, inicio=('el', 'gato')))

el gato duerme en el sofá el perro duerme en el sofá


# Ngrams model

In [1]:
import math
from collections import defaultdict

In [21]:
corpus = [
    "el gato duerme en la cama",
    "la gata duerme en el sofá",
    "el perro ladra fuerte",
]

In [22]:
tokenized = [sentence.lower().split() for sentence in corpus]

tokenized

[['el', 'gato', 'duerme', 'en', 'la', 'cama'],
 ['la', 'gata', 'duerme', 'en', 'el', 'sofá'],
 ['el', 'perro', 'ladra', 'fuerte']]

In [23]:
n = 3
ngram_counts = defaultdict(int)
context_counts = defaultdict(int)

for sentence in tokenized:
    sentence = ['<s>'] * (n-1) + sentence + ['</s>']
    for i in range(len(sentence) - n + 1):
        ngram = tuple(sentence[i:i+n])
        context = tuple(sentence[i:i+n-1])
        ngram_counts[ngram] += 1
        context_counts[context] += 1

In [24]:
ngram_counts

defaultdict(int,
            {('<s>', '<s>', 'el'): 2,
             ('<s>', 'el', 'gato'): 1,
             ('el', 'gato', 'duerme'): 1,
             ('gato', 'duerme', 'en'): 1,
             ('duerme', 'en', 'la'): 1,
             ('en', 'la', 'cama'): 1,
             ('la', 'cama', '</s>'): 1,
             ('<s>', '<s>', 'la'): 1,
             ('<s>', 'la', 'gata'): 1,
             ('la', 'gata', 'duerme'): 1,
             ('gata', 'duerme', 'en'): 1,
             ('duerme', 'en', 'el'): 1,
             ('en', 'el', 'sofá'): 1,
             ('el', 'sofá', '</s>'): 1,
             ('<s>', 'el', 'perro'): 1,
             ('el', 'perro', 'ladra'): 1,
             ('perro', 'ladra', 'fuerte'): 1,
             ('ladra', 'fuerte', '</s>'): 1})

In [25]:
context_counts

defaultdict(int,
            {('<s>', '<s>'): 3,
             ('<s>', 'el'): 2,
             ('el', 'gato'): 1,
             ('gato', 'duerme'): 1,
             ('duerme', 'en'): 2,
             ('en', 'la'): 1,
             ('la', 'cama'): 1,
             ('<s>', 'la'): 1,
             ('la', 'gata'): 1,
             ('gata', 'duerme'): 1,
             ('en', 'el'): 1,
             ('el', 'sofá'): 1,
             ('el', 'perro'): 1,
             ('perro', 'ladra'): 1,
             ('ladra', 'fuerte'): 1})

In [26]:
def get_prob(ngram):
    context = ngram[:-1]
    return ngram_counts[ngram] / context_counts[context] if context_counts[context] > 0 else 0.0

In [27]:
def perplexity(test_sentence, n):
    test_tokens = ['<s>'] * (n-1) + test_sentence.lower().split() + ['</s>']
    N = len(test_tokens) - n + 1
    log_prob = 0
    for i in range(N):
        ngram = tuple(test_tokens[i:i+n])
        prob = get_prob(ngram)
        if prob > 0:
            log_prob += math.log(prob)
        else:
            # Evita log(0) y penaliza fuertemente
            return float('inf')
    return math.exp(-log_prob / N)

In [50]:
frase = "el gato duerme en el sofá"
print("Perplexity:", perplexity(frase, 3))

Perplexity: 1.2917083420907467


In [51]:
frase2 = "el perro ladra en la cama"
print("Perplexity:", perplexity(frase2, 3))

Perplexity: inf


# Suavizado de Laplace

In [56]:
vocab = set()
for sentence in tokenized:
    for word in sentence:
        vocab.add(word)
vocab_size = len(vocab) + 1  # +1 para </s>

In [57]:
def get_prob_laplace(ngram):
    context = ngram[:-1]
    return (ngram_counts[ngram] + 1) / (context_counts[context] + vocab_size)

In [58]:
def perplexity_laplace(test_sentence, n):
    test_tokens = ['<s>'] * (n-1) + test_sentence.lower().split() + ['</s>']
    N = len(test_tokens) - n + 1
    log_prob = 0
    for i in range(N):
        ngram = tuple(test_tokens[i:i+n])
        prob = get_prob_laplace(ngram)
        log_prob += math.log(prob)
    return math.exp(-log_prob / N)

In [59]:
frase = "el gato duerme en el sillón"
print("Perplexity (Laplace):", perplexity_laplace(frase, 3))

Perplexity (Laplace): 7.706796894629428


In [61]:
frase2 = "el gato duerme en el sofa"
print("Perplexity (Laplace):", perplexity_laplace(frase2, 3))

Perplexity (Laplace): 7.706796894629428


# Add-k smoothing

In [64]:
# Función de probabilidad con add-k smoothing
def get_prob_add_k(ngram, k=0.01):
    context = ngram[:-1]
    return (ngram_counts[ngram] + k) / (context_counts[context] + k * vocab_size)

In [65]:
# Perplexity con add-k
def perplexity_add_k(test_sentence, n, k=0.01):
    test_tokens = ['<s>'] * (n-1) + test_sentence.lower().split() + ['</s>']
    N = len(test_tokens) - n + 1
    log_prob = 0
    for i in range(N):
        ngram = tuple(test_tokens[i:i+n])
        prob = get_prob_add_k(ngram, k)
        log_prob += math.log(prob)
    return math.exp(-log_prob / N)


In [66]:
# Ejemplo
frase = "el gato duerme en el sillón"
print("Perplexity (add-k=0.01):", perplexity_add_k(frase, 3, k=0.01))

Perplexity (add-k=0.01): 3.793374542808921


In [69]:
frase = "el gato duerme en la cama"
print("Perplexity (add-k=0.01):", perplexity_add_k(frase, 3, k=0.01))

Perplexity (add-k=0.01): 1.3961727522955707
